In [1]:
!pip install tensorflow_datasets
!pip install tensorflow


In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
# import seaborn as sns
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.image as mpimg
# import itertools

print(tf.__version__)

2.19.0


In [3]:
cifar10 = tf.keras.datasets.cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 18s 0us/step


In [4]:
# Shuffle the training data first
indices = tf.random.shuffle(tf.range(len(x_train)))
x_train_shuffled = tf.gather(x_train, indices)
y_train_shuffled = tf.gather(y_train, indices)

# Split into training and validation (80/20)
val_split = 0.2
num_train = int(len(x_train_shuffled) * (1 - val_split))
x_train_final = x_train_shuffled[:num_train]
y_train_final = y_train_shuffled[:num_train]
x_val = x_train_shuffled[num_train:]
y_val = y_train_shuffled[num_train:]

print(f"Training set size: {len(x_train_final)}")
print(f"Validation set size: {len(x_val)}")
print(f"Test set size: {len(x_test)}")

Training set size: 40000
Validation set size: 10000
Test set size: 10000


In [5]:
dataset, info = tfds.load("cifar10", as_supervised=True, with_info=True)
dataset_size = info.splits["train"].num_examples # 3670
class_names = info.features["label"].names # ["dandelion", "daisy", ...]
n_classes = info.features["label"].num_classes # 5

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cifar10/incomplete.1B1ZHE_3.0.2/cifar10-train.tfrecord*...:   0%|         …

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cifar10/incomplete.1B1ZHE_3.0.2/cifar10-test.tfrecord*...:   0%|          …

Dataset cifar10 downloaded and prepared to /root/tensorflow_datasets/cifar10/3.0.2. Subsequent calls will reuse this data.


In [6]:
model = tf.keras.applications.xception.Xception(weights="imagenet")

91884032/91884032 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step


In [7]:
sample_images = (x_train[:4].astype('float32')) / 255.0  # Normalize to 0-1
# Resize to 299x299 for Xception
images_resized = tf.image.resize(sample_images, (299, 299))

In [8]:
inputs = tf.keras.applications.xception.preprocess_input(images_resized)

In [9]:
batch_size = 32
"""
preprocess = tf.keras.Sequential([
tf.keras.layers.Resizing(height=299, width=299, crop_to_aspect_ratio=True),
tf.keras.layers.Lambda(tf.keras.applications.xception.preprocess_input)
])
train_set = tf.data.Dataset.from_tensor_slices((x_train_final, y_train_final)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)
valid_set = tf.data.Dataset.from_tensor_slices((x_val, y_val)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)
"""

preprocess = tf.keras.Sequential([
    tf.keras.layers.Resizing(height=299, width=299, crop_to_aspect_ratio=True),
    # This layer handles the -1 to 1 scaling correctly for Xception
    tf.keras.layers.Lambda(lambda x: tf.keras.applications.xception.preprocess_input(x))
])

train_set = tf.data.Dataset.from_tensor_slices((x_train_final, y_train_final)) \
    .map(lambda X, y: (preprocess(X), y)) \
    .batch(batch_size).prefetch(tf.data.AUTOTUNE)

valid_set = tf.data.Dataset.from_tensor_slices((x_val, y_val)) \
    .map(lambda X, y: (preprocess(X), y)) \
    .batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_set = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)

In [10]:
data_augmentation = tf.keras.Sequential([
tf.keras.layers.RandomFlip(mode="horizontal", seed=42),
tf.keras.layers.RandomRotation(factor=0.15, seed=42),
tf.keras.layers.RandomContrast(factor=0.45, seed=42),
tf.keras.layers.RandomZoom(height_factor=0.2, width_factor=0.2, seed=42),
tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1, seed=42),
])


In [11]:
base_model = tf.keras.applications.xception.Xception(weights="imagenet", include_top=False)

# 1. Define the Input
inputs = tf.keras.Input(shape=(299, 299, 3))

# 2. Add Data Augmentation
x = data_augmentation(inputs)

# 3. Pass through Xception
# Note: training=False keeps BatchNormalization stable during fine-tuning
x = base_model(x, training=False)

# 4. Rebuild the 'head' correctly
x = tf.keras.layers.GlobalAveragePooling2D()(x) # This processes the base_model output
x = tf.keras.layers.Dropout(0.70)(x)            # This applies dropout to the pooling
output = tf.keras.layers.Dense(n_classes, activation="softmax")(x)

# 5. Create the final Model object
model = tf.keras.Model(inputs=inputs, outputs=output)
"""
base_model = tf.keras.applications.xception.Xception(weights="imagenet", include_top=False)
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
dropout = tf.keras.layers.Dropout(0.70)(avg) # Start with 0.5
output = tf.keras.layers.Dense(n_classes, activation="softmax")(dropout)
model = tf.keras.Model(inputs=base_model.input, outputs=output)
"""

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


'\nbase_model = tf.keras.applications.xception.Xception(weights="imagenet", include_top=False)\navg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)\ndropout = tf.keras.layers.Dropout(0.70)(avg) # Start with 0.5\noutput = tf.keras.layers.Dense(n_classes, activation="softmax")(dropout)\nmodel = tf.keras.Model(inputs=base_model.input, outputs=output)\n'

In [ ]:
import time
for layer in base_model.layers[56:]:
    layer.trainable = True

optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1, # Be aggressive since you want to hit 0.11 quickly
    min_lr=1e-7
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,           # Wait 2 epochs to see if it improves again
    restore_best_weights=True # Extremely important: rolls back to the best version
)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
#history = model.fit(train_set, validation_data=valid_set, epochs=3)
start_time = time.time()
history_fine = model.fit(
    train_set,
    validation_data=valid_set,
    epochs=30,
    callbacks=[early_stop, lr_scheduler]
)
end_time = time.time()
print(f"Fine-tuning took: {end_time - start_time:.2f} seconds")

Epoch 1/30
 240/1250 ━━━━━━━━━━━━━━━━━━━━ 3:09 188ms/step - accuracy: 0.2834 - loss: 1.9861

In [ ]:
model.summary()